# Data Processing for Aalto Gear Failure Dataset (AGFD)

-   https://www.sciencedirect.com/science/article/pii/S2352340924011338
-   https://data.mendeley.com/datasets/fywnj597d8/2

**NOTES:**

`acc2` in installation 3 are pure noice, as the sensor is damaged. DO NOT USE!

**V1**

-   Step size 3

**V2**

-   Step size 2
-   Dropped `sample_index`


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import pyarrow as pa

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# HELPERS


def fix_time(X):
    increment = 1 / 3012  # For 3012 Hz sampling rate
    loc = np.where(X[1:] < X[:-1])[0] + 1  # + 1 needed to get the right index

    if len(loc) == 0:
        return X

    for l in loc:
        wrong = X[l - 1] - X[l]
        print(f"Wrong time at {l}")
        diff = np.concatenate(
            [
                np.zeros(l),
                np.ones(len(X) - l) * (increment + wrong),
            ]
        )
        X = X + diff

    return X

In [3]:
# SAMPLE OF DATA

data_folder = Path.cwd().parent.parent.parent / "data" / "mendeley_AGFD3kHz"
files = data_folder.glob("**/*.csv")

df = pd.read_csv(next(files), sep=",", index_col=0, header=0)
df.head(3)

,time,enc1_ang,enc2_ang,enc3_ang,enc4_ang,enc5_ang,acc1,acc2,acc3,acc4,Torq1,Torq2
0,0.000000,0.000,0.000,0.000,0.000,0.000,14.651109,-38.778650,-6.914053,-5.760087,9.780273,1.317871
1,0.000332,1.998,2.034,0.648,-0.666,0.162,79.047660,47.732230,1.118753,-4.851206,9.620667,1.144653
2,0.000664,4.014,4.032,1.332,-1.314,0.324,26.196007,36.811794,12.214314,30.963770,9.106751,1.509888


In [ ]:
# PROCESSING

torque_map = {
    "0_12Nm": 1,
    "0_71Nm": 6,
    "1_31Nm": 11,
}

# S - Small, L - Large
fault_map = {
    "Mild_Wear_GP7": ("S", "wear"),
    "Severe_Wear_GP2": ("L", "wear"),
    "Mild_TFF_GP9": ("S", "crack"),
    "Severe_TFF_GP5": ("L", "crack"),
    "Mild_Pitting_GP1": ("S", "pitting"),
    "Severe_Pitting_GP6": ("L", "pitting"),
}

# Specify raw data
data_folder = Path.cwd().parent.parent.parent / "data" / "mendeley_AGFD3kHz"
files = data_folder.glob("**/*.csv")

# Go through files
dfs = []
for f in files:
    # SPECS
    ##

    # Get measurement specifications from file path
    p = f.parts
    # print(p)

    rpm = int(p[6].replace("RPM", ""))
    torque = torque_map[p[7]]

    if p[8] == "Healthy":
        continue
    elif p[8] == "Faulty":
        installation = int(p[9][-1])
        level, fault = fault_map.get(
            p[10].replace(".csv", ""), (None, None)
        )  # Fault types we don't care about not in map
        GP = 0
    else:
        raise ValueError(f"Unknown type directory: {p[8]}")

    # Skip the faults we don't want
    if fault is None:
        continue

    # SIGNAL
    ##

    # Read CSV
    df = pd.read_csv(f, sep=",", index_col=0, header=0)

    # Installation 3 special case
    if installation == 3:
        # Reset index
        df = df.reset_index(
            drop=False  # ! Installation 3 has time as index. This is done to add separate index, but not drop time
        )

        # Cut first and last 20%, which has already been done for other installations
        df = df.iloc[int(len(df) * 0.2) : int(len(df) * 0.8)].reset_index(drop=True)

    print(str(f))

    # TODO: Save real average torque at faulty gear (Need to find out multiplier, because neither Torq1 or Torq2 are on the right shaft)
    # real_torque_avg = df["Torq1"].mean()

    # These sensors not used in this study
    df.drop(
        columns=[
            "time",
            "enc1_ang",
            "enc2_ang",
            "enc3_ang",
            "enc4_ang",
            "enc5_ang",
            # "acc1",
            # "Torq1",
            # "Torq2",
        ],
        inplace=True,
    )

    # Enc4 runs the wrong way in most files
    # if df["enc4_ang"].iloc[0] > df["enc4_ang"].iloc[1]:
    #     df["enc4_ang"] = -df["enc4_ang"]

    # Some measurements have glitches in time
    # df["time"] = fix_time(df["time"].to_numpy())

    # Sometimes last value is NaN, which causes problems later
    if any([np.isnan(df[s].iloc[-1]) for s in ["acc1", "acc2", "acc3", "acc4"]]):
        df = df.iloc[:-1].reset_index(drop=True)

    # Flip torques and standardize naming
    df = df.rename({"Torq2": "torque1", "Torq1": "torque2"}, axis="columns")

    # Get float 64 columns to flaot 32
    # * Conversion done because final computations are done with float32 anyway
    float64_cols = list(df.select_dtypes(include="float64"))
    df[float64_cols] = df[float64_cols].astype("float32")

    # Convert categoricals
    # dfs["severity"] = dfs["severity"].astype("category")
    # dfs["fault"] = dfs["fault"].astype("category")

    df["rpm"] = rpm
    df["torque"] = torque
    df["severity"] = level
    df["fault"] = fault
    # df["GP"] = GP
    df["installation"] = installation

    dfs.append(df)
    # print(df)
    # raise


print("Writing partition")
dfs = pd.concat(dfs)

dfs = dfs.reset_index(drop=True)
table = pa.Table.from_pandas(dfs)

partitions = ds.partitioning(
    pa.schema([("rpm", pa.int16()), ("installation", pa.int8())]), flavor="hive"
)
ds.write_dataset(
    table,
    Path.cwd().parent.parent.parent / "data" / "AGFD_osmalle",
    format="feather",
    partitioning=partitions,
    existing_data_behavior="overwrite_or_ignore",
)

/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_1/Severe_TFF_GP5.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_1/Mild_Pitting_GP1.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_1/Severe_Wear_GP2.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_1/Severe_Pitting_GP6.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_1/Mild_Wear_GP7.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_1/Mild_TFF_GP9.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_2/Severe_TFF_GP5.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_2/Mild_Pitting_GP1.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_2/Severe_Wear_GP2.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Inst_2/Severe_Pitting_GP6.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Faulty/Ins

In [12]:
# PROCESSING

torque_map = {
    "0_12Nm": 1,
    "0_71Nm": 6,
    "1_31Nm": 11,
}

# S - Small, L - Large
fault_map = {
    "Mild_Wear_GP7": ("S", "wear"),
    "Severe_Wear_GP2": ("L", "wear"),
    "Mild_TFF_GP9": ("S", "crack"),
    "Severe_TFF_GP5": ("L", "crack"),
    "Mild_Pitting_GP1": ("S", "pitting"),
    "Severe_Pitting_GP6": ("L", "pitting"),
}

# Specify raw data
data_folder = Path.cwd().parent.parent.parent / "data" / "mendeley_AGFD3kHz"
files = data_folder.glob("**/*.csv")

# Go through files
dfs = []
for f in files:
    # SPECS
    ##

    # Get measurement specifications from file path
    p = f.parts
    # print(p)

    rpm = int(p[6].replace("RPM", ""))
    torque = torque_map[p[7]]

    if p[8] == "Healthy":
        installation = 0
        level = "-"
        GP = int(p[9][2])
        fault = "baseline"
    elif p[8] == "Faulty":
        continue
    else:
        raise ValueError(f"Unknown type directory: {p[8]}")

    # Skip the faults we don't want
    if fault is None:
        continue

    # SIGNAL
    ##

    # Read CSV
    df = pd.read_csv(f, sep=",", index_col=0, header=0)

    # Installation 3 special case
    if installation == 3:
        # Reset index
        df = df.reset_index(
            drop=False  # ! Installation 3 has time as index. This is done to add separate index, but not drop time
        )

        # Cut first and last 20%, which has already been done for other installations
        df = df.iloc[int(len(df) * 0.2) : int(len(df) * 0.8)].reset_index(drop=True)

    print(str(f))

    # TODO: Save real average torque at faulty gear (Need to find out multiplier, because neither Torq1 or Torq2 are on the right shaft)
    # real_torque_avg = df["Torq1"].mean()

    # These sensors not used in this study
    df.drop(
        columns=[
            "time",
            "enc1_ang",
            "enc2_ang",
            "enc3_ang",
            "enc4_ang",
            "enc5_ang",
            # "acc1",
            # "Torq1",
            # "Torq2",
        ],
        inplace=True,
    )

    # Enc4 runs the wrong way in most files
    # if df["enc4_ang"].iloc[0] > df["enc4_ang"].iloc[1]:
    #     df["enc4_ang"] = -df["enc4_ang"]

    # Some measurements have glitches in time
    # df["time"] = fix_time(df["time"].to_numpy())

    # Sometimes last value is NaN, which causes problems later
    if any([np.isnan(df[s].iloc[-1]) for s in ["acc1", "acc2", "acc3", "acc4"]]):
        df = df.iloc[:-1].reset_index(drop=True)

    # Flip torques and standardize naming
    df = df.rename({"Torq2": "torque1", "Torq1": "torque2"}, axis="columns")

    # Get float 64 columns to flaot 32
    # * Conversion done because final computations are done with float32 anyway
    float64_cols = list(df.select_dtypes(include="float64"))
    df[float64_cols] = df[float64_cols].astype("float32")

    # Convert categoricals
    # dfs["severity"] = dfs["severity"].astype("category")
    # dfs["fault"] = dfs["fault"].astype("category")

    df["rpm"] = rpm
    df["torque"] = torque
    df["severity"] = GP
    df["fault"] = fault
    # df["GP"] = GP
    # df["installation"] = installation

    dfs.append(df)
    # print(df)
    # raise


print("Writing partition")
dfs = pd.concat(dfs)

dfs = dfs.reset_index(drop=True)
table = pa.Table.from_pandas(dfs)

partitions = ds.partitioning(pa.schema([("rpm", pa.int16())]), flavor="hive")

ds.write_dataset(
    table,
    Path.cwd().parent.parent.parent / "data" / "AGFD_baseline_osmalle",
    format="feather",
    partitioning=partitions,
    existing_data_behavior="overwrite_or_ignore",
)

/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP9.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP3.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP2.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP1.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP5.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP4.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP6.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/0_71Nm/Healthy/GP7.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/1_31Nm/Healthy/GP9.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/1_31Nm/Healthy/GP3.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/1_31Nm/Healthy/GP2.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/1_31Nm/Healthy/GP1.csv
/Users/hamalaa14/git/data/mendeley_AGFD3kHz/1000RPM/1_31Nm/Healthy/GP5.csv
/Users/hamalaa14/git/data